# 2026 World Cup — tracing an Elo forecast from ratings to charts

**Historical reconstruction**, originally produced in June 2026; audited September 2026. This is not a post-tournament reforecast. Run the cells in order from this package root. For the full provenance and discrepancy table, see [`AUDIT.md`](AUDIT.md).

## 1 · Why are there two different prediction tables?

An early notebook used **generic 1-v-32 seeding** and 1,000 trials; its own CSV estimates Spain at **21.7%** and Argentina **21.2%**. A later visual pipeline used FIFA-published Round-of-32 eligibility constraints, deterministic backtracking instead of the exact Annex C, and **600 trials** (seed **2026**). The latter produced Argentina **21.5%**, Spain **19.33%**, and France **12.5%**. The deck iterations mixed these figures and sometimes described 1,000 trials while displaying 600-trial results. We keep the second run as the reproducible visual baseline and never pool the two. Neither is a verified official-bracket model.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import world_cup_simulation as sim
BASE = Path.cwd()
ratings, elo = sim.load_ratings(BASE / "data" / "optimized_elo_2026.csv")
assert len(ratings) == 48 and ratings.Team.is_unique
ratings.head(8)

## 2 · From match history to Elo: an important missing input

The original `legacy/original_team_ratings.ipynb` downloaded `martj42/international_results/master/results.csv` and optimized match-type K-factors using in-sample historical squared expected-score errors from 2000 onward. Reported fitted K-factors were **27.09 friendly / 30.77 qualifier / 27.41 continental / 37.61 World Cup**. The exact underlying data commit and training cutoff are **not recorded**. Re-running the original training notebook now could introduce future match results. For a legitimate historical replay, we start from its **frozen** 48-team ratings, not a new network download. This is reproduction of the tournament simulation, **not independent reproduction or validation of the Elo fit**.

In [ ]:
ratings.describe(include='all').T

## 3 · What does Elo actually mean?\n\nFor ratings $R_A,R_B$, expected match score for A is $E_A=1/(1+10^{-(R_A-R_B)/400})$. An expected score averages a win as 1, a draw as 0.5, a loss as 0. The tournament simulator **heuristically** chooses $p_D=0.18+0.12e^{-|R_A-R_B|/300}$ (clipped to 0.16–0.32), then sets $p_A=(1-p_D)E_A$, $p_B=(1-p_D)(1-E_A)$. This preserves $p_A+p_D+p_B=1$, but does **not guarantee** the three-way distribution is calibrated. The script also samples scoreline margins from fixed distributions rather than fitting a goals model.

In [ ]:
for a,b in [('Spain','Argentina'),('France','Haiti')]:
    print(a, 'vs', b, 'expected score:', round(sim.elo_expected_score(a,b,elo),3),
          'win/draw/loss:', tuple(round(v,3) for v in sim.match_probs(a,b,elo)))

## 4 · One full tournament: a **single illustrative realization**, not the forecast

There are 12 groups × 6 matches = 72 group games. The top two from each group plus the best eight third-place finishers advance. Here a fixed seed makes a traceable example. Group and third-place tiebreakers use Elo as a last resort instead of all FIFA rules; third-place slots follow eligible group constraints without FIFA Annex C's predetermined 495-case mapping.

In [ ]:
standings, qualifiers, thirds, slots, bracket, example_champion = sim.simulate_tournament(elo, seed=14)
assert (len(standings), len(qualifiers), len(thirds), len(slots), len(bracket)) == (48,32,12,16,31)
print('Example-only champion:', example_champion)
display(standings[standings.Group == 'J'][['Pos','Team','Pts','GD','GF']])
display(slots[['Match','Slot A','Team A','Slot B','Team B']].head())

## 5 · Reproduce the exact archived 600-trial output

Set **600** and **2026** below to reproduce the June 2026 visual pipeline. For a *new exploratory experiment*, you can change the seed or sample size, but the archived graphics and percentages will no longer be exact matches. Counts divided by 600 produce all reported stage frequencies.

In [ ]:
stage, title = sim.run_monte_carlo(elo, n_sims=600, seed=2026)
archived = pd.read_csv(BASE / 'data' / 'legacy_monte_carlo_champion_probabilities.csv')
check = title.merge(archived, on='Team', suffixes=('_rerun','_archived'))
assert np.allclose(check['Champion %_rerun'], check['Champion %_archived'])
assert np.isclose(title['Champion %'].sum(),1)
print('Reproduced all 48 archived title frequencies exactly.')
title[['Team','Champion %','Final %','Qualified %']].head(12)

## 6 · Sampling uncertainty is not model accuracy\n\nFor a simulated frequency $\hat p=k/n$, the rough binomial standard error is $\sqrt{\hat p(1-\hat p)/n}$. For the Argentina 600-run frequency of 21.5%, this is about 1.68 **percentage points**, before considering team-outcome correlation, data issues, heuristic scores, bracket approximation or systematic model error. This is a Monte Carlo precision estimate, not a confidence interval for a well-calibrated forecast.

In [ ]:
arg_p=float(title.set_index('Team').loc['Argentina','Champion %'])
n=600
print('Argentina:',f'{100*arg_p:.2f}%', 'approximate Monte Carlo SE:',f'{100*np.sqrt(arg_p*(1-arg_p)/n):.2f} percentage points')
ax=title.head(12).iloc[::-1].plot.barh(x='Team',y='Champion %',legend=False,figsize=(9,5))
ax.set(xlabel='Fraction of archived simulated tournaments',title='600-run historical experiment (not validated probabilities)')
plt.tight_layout()

## 7 · What did the original charts show?

The original eight PNG figures (in `outputs/`) were checked against the numeric run: one group-stage realization, the best third-place race, expected group points/goal difference, advancement distributions, champion frequencies, and an illustrative approximate-slot bracket. **A plotted champion path is an example scenario, not a prediction that specific matches will happen.**

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(BASE/'outputs'/'06_champion_probability_ranking.png'),width=800))

## 8 · How did the real tournament compare?

FIFA records Spain 1–0 Argentina in the actual July 19, 2026 final. Spain received 19.33% of the archived 600-run *model frequency*, and Argentina 21.5%; the recorded result does not alone prove or disprove calibration. We do not compute retrospective match-level Brier scores because the original model did not archive pre-kickoff per-match probability predictions against a versioned complete actual-results dataset. Such a comparison would require clearly separating out-of-sample predictions from observed outcomes.

**Sources:** [FIFA actual tournament summary](https://www.fifa.com/en/tournaments/mens/worldcup/2026/articles/knockout-stage-match-schedule-bracket), [FIFA format](https://gpcustomersupportfwc2026.tickets.fifa.com/hc/en-gb/articles/28784798873117-9-What-is-the-format-for-the-FIFA-World-Cup-2026-tournament), [historical match dataset](https://github.com/martj42/international_results).

## 9 · Limitations and next iteration

The priority upgrades are (a) freeze raw training data and dates, (b) perform time-respecting out-of-sample evaluation, (c) estimate 3-way probabilities and goal distribution on held-out data, (d) implement all competition tiebreakers and FIFA Annex C, and (e) save every pre-match probability with provenance before scoring the model. See `AUDIT.md`. The reusable implementation and command-line interface are in `world_cup_simulation.py`.